In [ ]:
# Finalspark header
import numpy as np
import time
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output
from neuroplatform import StimShape, StimParam , IntanSofware, Trigger, Database, StimPolarity, Experiment

import pandas as pd
from dateutil import parser
from lets_plot import *
from tqdm import tqdm
LetsPlot.setup_html()

In [ ]:
# Establish API credentials
token = "<insert token here>"
exp = Experiment(token)
print(f'Electrodes: {exp.electrodes}') # Electrodes that you can used


In [ ]:
# Establish db connection and get exp name
db = Database()
exp_name = exp.exp_name
exp_name

In [ ]:
# Get current time func
def get_current_utc_time():
    """Returns the current UTC time as a datetime object."""
    return datetime.now(timezone.utc)

In [ ]:
# Start time parser - Timestamps gained from experiment protocols
s1 = datetime(2024, 5, 2, 9, 0, 0, 941232)
s1

# End time parser
s2 = datetime(2024, 5, 2, 12, 0, 0, 956840)
s2

In [ ]:
# Trigger data collection point
triggers = db.get_all_triggers(s1, s2)
triggers

In [ ]:
# SPM data collection point
df_spike_count = db.get_spike_count(s1, s2, exp_name)
df_spike_count

In [ ]:
# Raw spike data collection point
df_spike_event = db.get_spike_event(s1,s2,exp_name)
df_spike_event

## Below are several plotting features that can be run. For large datasets batch processing is required for effective raw-data analysis


In [ ]:
# Plot SPM
def plot_df(df: pd.DataFrame, title: str):
    return ggplot(df, aes(x='Time', y='Spike per minutes', group='channel', color='channel')) + ggsize(800, 600) + geom_line() + \
    ggtitle(title)

plot_df(df_spike_count, exp_name)

In [ ]:
# Raw waveform over each channel
'''
WARNING: DO NOT USE LARGE DATASETS: MAXIMUM EXPERIMENT DURATION ~10MINS SAFELY

Needs breaking up into batch processing based on temporal codes to allow Finalspark server to not be overloaded
'''
raws = []
for i, row in tqdm(df_spike_event.iterrows(), total=df_spike_event.shape[0]):
    t = row['Time']
    t1 = t-timedelta(milliseconds=1)
    t2 = t+timedelta(milliseconds=2)
    raws.append(db.get_raw_spike(t1,t2, row['channel']))

def plot_raw_channel(channel: int):
    df_chan = df_spike_event[df_spike_event['channel']==channel]
    # Start with an empty plot
    p = ggplot()
    p += ggtitle(f'Channel {channel}')

    for i, row in df_chan.iterrows():
        df_raw = raws[i]
        if df_raw.shape[0] == 90:
            df_raw['Time [ms]'] = np.linspace(-1,2,90)
            p += geom_line(aes(x='Time [ms]', y='Amplitude'), data=df_raw)

    return p

channels = df_spike_event['channel'].unique()
list_plot = []
for i,chan in enumerate(channels):
    list_plot.append(plot_raw_channel(chan))

nbCol = 3
nbRow = len(list_plot)//nbCol
if len(list_plot)%nbCol != 0:
    nbRow += 1
gggrid(list_plot, ncol=nbCol) + ggsize(800, 300*nbRow)